# C06 组合优化与课程收官（在线版）

使用在线 ETF 数据做组合优化，并与等权对比。


In [ ]:
START_DATE = "2019-01-01"
END_DATE = "2024-12-31"
UNIVERSE = ["510300.XSHG", "510500.XSHG", "159915.XSHE", "512100.XSHG", "518880.XSHG", "511010.XSHG"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import rqdatac

# 方式1（推荐课堂演示）：把教育版 license 直接粘贴到 PASSWD
PASSWD = "在这里粘贴你的教育版license"

# 方式2（不想明文写在 notebook）：注释掉上面一行，改用环境变量
# import os
# PASSWD = os.getenv("RQDATAC_LICENSE", "")

if PASSWD and ("在这里粘贴" not in PASSWD):
    rqdatac.init('license', PASSWD)
    print("rqdatac 初始化成功")
else:
    print("请先填写 PASSWD，再运行本单元")


In [ ]:
close = rqdatac.get_price(
    UNIVERSE,
    start_date=START_DATE,
    end_date=END_DATE,
    fields="close",
    adjust_type="pre",
    expect_df=False,
)
close = close.droplevel(0, axis=1) if isinstance(close.columns, pd.MultiIndex) else close

ret_df = close.resample("ME").last().pct_change().dropna()
ret_df.head()


## 夏普最大化权重


In [ ]:
from scipy.optimize import minimize


def optimize_sharpe(mean_ret, cov_mat, rf=0.0):
    n = len(mean_ret)

    def objective(w):
        port_ret = w @ mean_ret
        port_vol = np.sqrt(w @ cov_mat @ w)
        return -(port_ret - rf) / (port_vol + 1e-12)

    x0 = np.ones(n) / n
    bounds = [(0.0, 1.0)] * n
    constraints = [{"type": "eq", "fun": lambda w: w.sum() - 1.0}]

    res = minimize(objective, x0=x0, bounds=bounds, constraints=constraints)
    return res.x

w_opt = optimize_sharpe(ret_df.mean().values, ret_df.cov().values)
w_eq = np.ones(len(UNIVERSE)) / len(UNIVERSE)
weights = pd.DataFrame({"asset": UNIVERSE, "w_opt": w_opt, "w_eq": w_eq}).round(4)
weights


## 滚动优化回测


In [ ]:
lookback = 12
dates_bt, r_opt, r_eq = [], [], []

for t in range(lookback, len(ret_df) - 1):
    hist = ret_df.iloc[t - lookback:t]
    nxt = ret_df.iloc[t + 1]
    w = optimize_sharpe(hist.mean().values, hist.cov().values)

    dates_bt.append(ret_df.index[t + 1])
    r_opt.append(float(w @ nxt.values))
    r_eq.append(float(nxt.mean()))

bt = pd.DataFrame({"optimized": r_opt, "equal_weight": r_eq}, index=dates_bt)
nav = (1 + bt).cumprod()

fig, ax = plt.subplots(figsize=(10, 4))
nav.plot(ax=ax, title="C06 在线版：组合优化 vs 等权")
ax.set_ylabel("NAV")
plt.show()


def perf(r):
    ann_ret = r.mean() * 12
    ann_vol = r.std() * np.sqrt(12)
    sharpe = ann_ret / (ann_vol + 1e-12)
    mdd = ((1 + r).cumprod() / (1 + r).cumprod().cummax() - 1).min()
    return ann_ret, ann_vol, sharpe, mdd

summary = pd.DataFrame(
    [perf(bt["optimized"]), perf(bt["equal_weight"])],
    columns=["annual_return", "annual_vol", "sharpe", "max_drawdown"],
    index=["optimized", "equal_weight"],
).round(4)
summary


## 收官
全流程已切换为在线数据版本，可直接用于课堂实时演示。
